# WeakSeg: Annotation-Efficient Semantic Segmentation with Image-Level Supervision

Self-contained, GPU-accelerated entrypoint for **WeakSeg** on Kaggle. One run executes the full comparison:

- **Fully supervised baseline** - DeepLabV3-ResNet50 trained on pixel masks.
- **Naive CAM baseline** - ResNet-50 multi-label classifier -> class activation maps -> thresholded pseudo-masks -> DeepLabV3.
- **CAM + DenseCRF** - the same pseudo-masks refined with DenseCRF (pseudo-label quality experiment).
- **SEAM refinement** - equivariant-attention training plus Pixel Correlation Module affinity propagation (Wang et al., CVPR 2020) -> expanded pseudo-masks -> DeepLabV3.

Weak pipelines see **image-level labels only** (`ImageSets/Main`); VOC pixel masks are used exclusively for final evaluation.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

WORKDIR = Path('/kaggle/working/weak-seg')
REPO_URL = 'https://github.com/kushc2004/weak-seg.git'

if WORKDIR.is_dir() and (WORKDIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=False)
elif WORKDIR.is_dir() and (WORKDIR / 'src' / 'weakseg').is_dir():
    print('Using existing local workspace copy.')
else:
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
    except Exception as e:
        print(f'Could not clone repository ({e}). Expecting an attached workspace copy.')
        WORKDIR = Path('/kaggle/working')

os.chdir(WORKDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-build-isolation', '-q'], check=True)
print('Working directory:', WORKDIR)

In [ ]:
# DenseCRF support for the pseudo-label refinement experiment.
# pydensecrf has no modern wheels: try PyPI forks first, then build from source.
import subprocess, sys

def crf_available():
    try:
        import pydensecrf.densecrf  # noqa: F401
        return True
    except Exception:
        return False

if not crf_available():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydensecrf2'], check=False)
if not crf_available():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cython'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/lucasb-eyer/pydensecrf.git'], check=False)
print('DenseCRF available:', crf_available())

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print(f'Using CUDA GPU: {gpu_name} (Capability: {capability})')
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('Using Apple MPS GPU.')
    DEVICE = 'mps'
else:
    print('No GPU detected; running on CPU.')
    DEVICE = 'cpu'
print('Selected Device:', DEVICE)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

# Resume support: attach a previous artifact bundle OR a raw salvage cache
# (e.g. classifier checkpoints recovered from an errored session).
input_root = Path('/kaggle/input')
archives = sorted(input_root.rglob('weakseg_artifacts.tar.gz'))
cache_restored = False

if archives:
    print(f'Found derived artifact cache: {archives[0]}')
    subprocess.run([sys.executable, 'scripts/restore_kaggle_artifacts.py', str(archives[0])], check=True)
    cache_restored = True
else:
    raw_caches = sorted(input_root.rglob('outputs/checkpoints'))
    if raw_caches:
        source_outputs = raw_caches[0].parent          # .../outputs
        print(f'Found raw artifact cache: {source_outputs}')
        shutil.copytree(source_outputs, WORKDIR / 'outputs', dirs_exist_ok=True)
        cache_restored = True

if cache_restored:
    print('Cache restored - completed stages with intact artifacts will be SKIPPED.')
else:
    print('No previous cache attached; running full pipeline from initial stages.')

In [ ]:
import subprocess
import sys
from pathlib import Path

# If any cache was restored above, DON'T force: completed stages (with their
# checkpoints present) are skipped, so you only retrain what you deleted or changed.
cache_restored = bool(sorted(Path('/kaggle/input').rglob('weakseg_artifacts.tar.gz'))
                      or sorted(Path('/kaggle/input').rglob('outputs/checkpoints')))
cmd = [
    sys.executable, 'scripts/run_full_pipeline.py',
] + ([] if cache_restored else ['--force']) + [
    f'device={DEVICE}',
    'train_list=train',       # switch to train_aug when SegmentationClassAug is available
]
print('Executing:', ' '.join(cmd))
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(f'Pipeline failed with exit code {result.returncode}')

In [ ]:
from IPython.display import Markdown, display
import json

report_path = WORKDIR / 'outputs/reports/experiment_summary.md'
if report_path.is_file():
    display(Markdown(report_path.read_text(encoding='utf-8')))

state_path = WORKDIR / 'outputs/pipeline_state.json'
if state_path.is_file():
    print('\nPipeline Execution State:')
    state = json.loads(state_path.read_text())
    print(json.dumps({name: stage.get('status') for name, stage in state['stages'].items()}, indent=2))

In [ ]:
from IPython.display import Image as IPyImage, display

viz_dir = WORKDIR / 'outputs/visualizations'
grids = sorted(viz_dir.glob('*.jpg'))[:10]
for grid in grids:
    print(grid.name)
    display(IPyImage(filename=str(grid)))

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

# Bundle outputs so the next session can resume and results can be downloaded.
subprocess.run([sys.executable, 'scripts/publish_kaggle_artifacts.py', '--no-upload'], check=True)

out_working = Path('/kaggle/working')
archive = WORKDIR / 'artifacts/kaggle/weakseg_artifacts.tar.gz'
manifest = WORKDIR / 'outputs/kaggle_artifact_manifest.json'
if archive.is_file():
    shutil.copy2(archive, out_working / 'weakseg_artifacts.tar.gz')
if manifest.is_file():
    shutil.copy2(manifest, out_working / 'weakseg_artifact_manifest.json')

results_md = WORKDIR / 'RESULTS.md'
if results_md.is_file():
    shutil.copy2(results_md, out_working / 'RESULTS.md')
print('Artifact bundle ready at:', out_working / 'weakseg_artifacts.tar.gz')